In [1]:
import math

import torch
import torch.nn as nn

torch.manual_seed(42)

# Une phrase de 4 mots, deja "embeddee" (dimension 8)
phrase = torch.randn(1, 4, 8)
mots = ["the", "delivery", "was", "terrible"]

d_model = 8
d_k = 8  # dimension des projections Q, K, V (simplifie ici, egal a d_model)

W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

Q = W_q(phrase)
K = W_k(phrase)
V = W_v(phrase)

print("Formes de Q, K, V :", Q.shape, K.shape, V.shape)
print("-> chaque mot a maintenant 3 versions de lui-meme\n")

# Etape 1 : scores de compatibilite Q . K^T
scores = Q @ K.transpose(-2, -1)
print("Matrice de scores BRUTS (4x4, avant mise a l'echelle) :")
print(scores[0].detach().numpy().round(2))

# Etape 2 : mise a l'echelle
scores_scaled = scores / math.sqrt(d_k)
print("\nApres mise a l'echelle (/ sqrt(", d_k, ")) :")
print(scores_scaled[0].detach().numpy().round(2))

Formes de Q, K, V : torch.Size([1, 4, 8]) torch.Size([1, 4, 8]) torch.Size([1, 4, 8])
-> chaque mot a maintenant 3 versions de lui-meme

Matrice de scores BRUTS (4x4, avant mise a l'echelle) :
[[ 1.25  0.04 -0.41  0.6 ]
 [-0.66  0.11 -0.19  0.7 ]
 [ 0.35 -0.6  -0.57 -2.45]
 [-0.29  0.97  1.86  0.03]]

Apres mise a l'echelle (/ sqrt( 8 )) :
[[ 0.44  0.01 -0.14  0.21]
 [-0.23  0.04 -0.07  0.25]
 [ 0.12 -0.21 -0.2  -0.87]
 [-0.1   0.34  0.66  0.01]]


In [7]:
import math

import torch
import torch.nn as nn

torch.manual_seed(42)

d_model = 8  # dimension totale
n_heads = 2  # 2 tetes en parallele
d_k = d_model // n_heads  # chaque tete travaille sur une portion plus petite

print(f"d_model={d_model}, n_heads={n_heads} -> chaque tete a d_k={d_k}")

phrase = torch.randn(1, 4, d_model)  # 4 mots

W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model, bias=False)  # projection finale

Q = W_q(phrase)  # (1, 4, 8)
K = W_k(phrase)
V = W_v(phrase)


# ETAPE CLE : decouper la dimension 8 en 2 tetes de 4
def decouper_en_tetes(x, n_heads, d_k):
    batch, longueur, _ = x.shape
    x = x.view(batch, longueur, n_heads, d_k)
    return x.permute(0, 2, 1, 3)  # (batch, n_heads, longueur, d_k)


Q_tetes = decouper_en_tetes(Q, n_heads, d_k)
K_tetes = decouper_en_tetes(K, n_heads, d_k)
V_tetes = decouper_en_tetes(V, n_heads, d_k)

print("Forme de Q apres decoupage en tetes :", Q_tetes.shape)
print("-> (batch, 2 tetes, 4 mots, 4 dimensions par tete)")

d_model=8, n_heads=2 -> chaque tete a d_k=4
Forme de Q apres decoupage en tetes : torch.Size([1, 2, 4, 4])
-> (batch, 2 tetes, 4 mots, 4 dimensions par tete)


## 1.   Attention   mechanism

In [9]:
# Prerequis : uv add torch
# Utilise les fonctions reelles de src/transformer_arch/attention_mechanism.py

# --- BLOC 1 : self-attention de base, matrice de poids ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import torch

from transformers_arch.attention_mechanism import (
    MultiHeadAttention,
    SelfAttention,
    positional_encoding,
    scaled_dot_product_attention,
)

torch.manual_seed(42)
mots = ["the", "delivery", "was", "terrible"]
phrase = torch.randn(1, 4, 8)

model = SelfAttention(d_model=8)
sortie, poids = model(phrase)

print("Matrice des poids d'attention :\n")
print(" " * 12 + "".join(f"{m:>10}" for m in mots))
for i, ligne in enumerate(mots):
    valeurs = " ".join(f"{v:9.3f}" for v in poids[0, i].detach().numpy())
    print(f"{ligne:12}{valeurs}")

print("\nSomme de chaque ligne :", poids[0].sum(dim=-1).detach().numpy().round(4))

Matrice des poids d'attention :

                   the  delivery       was  terrible
the             0.333     0.217     0.185     0.265
delivery        0.196     0.257     0.231     0.316
was             0.356     0.255     0.257     0.132
terrible        0.172     0.268     0.367     0.193

Somme de chaque ligne : [1. 1. 1. 1.]


In [10]:
# --- BLOC 2 : preuve -- le self-attention seul est invariant par ---
# --- permutation (aucune notion d'ordre) ---
torch.manual_seed(42)
model2 = SelfAttention(d_model=8)

mot_a = torch.randn(1, 1, 8)
mot_b = torch.randn(1, 1, 8)
mot_c = torch.randn(1, 1, 8)

phrase_1 = torch.cat([mot_a, mot_b, mot_c], dim=1)  # a b c
phrase_2 = torch.cat([mot_c, mot_b, mot_a], dim=1)  # c b a

sortie_1, _ = model2(phrase_1)
sortie_2, _ = model2(phrase_2)

print("\nSortie 'a' dans 'a b c' :", sortie_1[0, 0].detach().numpy().round(3))
print("Sortie 'a' dans 'c b a' :", sortie_2[0, 2].detach().numpy().round(3))
print("Identiques ?", torch.allclose(sortie_1[0, 0], sortie_2[0, 2], atol=1e-5))
print("-> confirme : sans position, l'ordre n'a aucune influence")


Sortie 'a' dans 'a b c' : [ 0.245  0.421  0.2    0.009 -0.627 -0.103 -0.195 -0.146]
Sortie 'a' dans 'c b a' : [ 0.245  0.421  0.2    0.009 -0.627 -0.103 -0.195 -0.146]
Identiques ? True
-> confirme : sans position, l'ordre n'a aucune influence


In [14]:
# --- BLOC 3 : le positional encoding casse cette symetrie ---
pe = positional_encoding(max_len=3, d_model=8).unsqueeze(0)

sortie_1_pe, _ = model2(phrase_1 + pe)
sortie_2_pe, _ = model2(phrase_2 + pe)

print("\nAvec positional encoding ajoute :")
print("Identiques ?", torch.allclose(sortie_1_pe[0, 0], sortie_2_pe[0, 2], atol=1e-5))
print("-> maintenant different : le modele distingue la position")


Avec positional encoding ajoute :
Identiques ? False
-> maintenant different : le modele distingue la position


In [15]:
# --- BLOC 4 : masquage avec -inf, verifier le poids exact ---
q = torch.randn(1, 3, 4)
k = torch.randn(1, 3, 4)
v = torch.randn(1, 3, 4)

mask = torch.zeros(1, 3, 3, dtype=torch.bool)
mask[0, :, 2] = True  # masque toute la colonne 2 (ex: position de padding)

_, poids_masques = scaled_dot_product_attention(q, k, v, mask)
print("\nPoids avec la colonne 2 masquee :")
print(poids_masques[0].detach().numpy().round(3))
print("Somme de chaque ligne :", poids_masques[0].sum(dim=-1).numpy().round(4))
print("-> la colonne masquee est exactement 0, le reste se renormalise a 1")


Poids avec la colonne 2 masquee :
[[0.643 0.357 0.   ]
 [0.768 0.232 0.   ]
 [0.494 0.506 0.   ]]
Somme de chaque ligne : [1. 1. 1.]
-> la colonne masquee est exactement 0, le reste se renormalise a 1


In [16]:
# --- BLOC 5 : multi-head -- tetes differentes, poids differents ---
torch.manual_seed(42)
mha = MultiHeadAttention(d_model=8, n_heads=2)
sortie_mha, poids_mha = mha(phrase)

print("\nForme des poids multi-head :", poids_mha.shape)
print("-> (batch, n_heads, longueur, longueur)")

tete_1 = poids_mha[0, 0]
tete_2 = poids_mha[0, 1]
print("\nTete 1 :\n", tete_1.detach().numpy().round(3))
print("Tete 2 :\n", tete_2.detach().numpy().round(3))
print("Identiques ?", torch.allclose(tete_1, tete_2, atol=1e-4))


Forme des poids multi-head : torch.Size([1, 2, 4, 4])
-> (batch, n_heads, longueur, longueur)

Tete 1 :
 [[0.203 0.26  0.312 0.225]
 [0.374 0.252 0.092 0.282]
 [0.322 0.163 0.35  0.166]
 [0.321 0.156 0.352 0.171]]
Tete 2 :
 [[0.332 0.307 0.156 0.206]
 [0.185 0.238 0.18  0.397]
 [0.431 0.197 0.23  0.142]
 [0.255 0.236 0.309 0.2  ]]
Identiques ? False
